# 01 Data Acquisition

Notebook นี้แสดงกระบวนการเก็บข้อมูลทั้งหมดโดยตรง ตั้งแต่การอ่าน MasterData การส่ง request รายเดือน การดาวน์โหลดข้อมูลภายนอก การตรวจความครบถ้วน ไปจนถึงการบันทึกแหล่งข้อมูลและ checksum

แหล่งข้อมูลประกอบด้วย

- ดัชนีราคาสินค้านำเข้า (IMI) จากสำนักงานนโยบายและยุทธศาสตร์การค้า
- อัตราแลกเปลี่ยนบาทต่อดอลลาร์ `DEXTHUS` จาก FRED
- ราคาน้ำมัน Brent `DCOILBRENTEU` จาก FRED

ข้อมูลใน `data/raw/` ไม่มีการแก้ไขหรือเขียนทับ หากต้องการดาวน์โหลดใหม่ให้เปลี่ยน `DOWNLOAD_NEW` เป็น `True` ระบบจะสร้างชื่อไฟล์ใหม่ด้วย timestamp UTC

## 1. การตั้งค่าและฟังก์ชันพื้นฐาน

ค่าเริ่มต้นใช้ snapshot ที่ดาวน์โหลดและตรวจสอบแล้ว เพื่อป้องกันการยิง API ซ้ำเมื่อกด Run All ระหว่างตรวจงาน หากต้องการดึงข้อมูลใหม่จริงให้กำหนด `DOWNLOAD_NEW = True`

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DOWNLOAD_NEW = False
REQUEST_DELAY_SECONDS = 0.30

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'PROJECT_SCOPE.md').exists()
)
RAW_DIR = ROOT / 'data' / 'raw'
SOURCES_PATH = RAW_DIR / 'sources.json'
MASTER_URL = 'https://index-api.tpso.go.th/OpenApi/Imex/Month/MasterData'
MONTH_URL = 'https://index-api.tpso.go.th/OpenApi/Imex/Month'
FRED_SERIES = {
    'DEXTHUS': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXTHUS',
    'DCOILBRENTEU': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU',
}

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def make_session() -> requests.Session:
    retry = Retry(
        total=5, connect=5, read=5, backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET', 'POST'}),
        respect_retry_after_header=True,
    )
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers['User-Agent'] = 'imi-early-warning/1.0 (academic project)'
    return session

def enumerate_months(start_year, start_month, end_year, end_month):
    result = []
    year, month = start_year, start_month
    while (year, month) <= (end_year, end_month):
        result.append((year, month))
        month += 1
        if month == 13:
            year, month = year + 1, 1
    return result

session = make_session()
downloaded_at = datetime.now(timezone.utc)
stamp = downloaded_at.strftime('%Y%m%dT%H%M%SZ')
print(f'Project root: {ROOT}')
print(f'Download new data: {DOWNLOAD_NEW}')

Project root: /Users/pattarapol/itkmitl/imi-early-warning
Download new data: False


## 2. อ่าน MasterData ของ IMI

MasterData เป็นตัวกำหนดปีฐาน เดือนเริ่มต้น เดือนล่าสุด และรายชื่อรหัสสินค้า จึงไม่กำหนดข้อมูลเหล่านี้แบบ hard-code

In [2]:
sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))

if DOWNLOAD_NEW:
    response = session.get(MASTER_URL, timeout=60)
    response.raise_for_status()
    master_data = response.json()
    master_path = RAW_DIR / f'imi_master_{stamp}.json'
    master_path.write_text(
        json.dumps(master_data, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
else:
    imi_source = next(
        item for item in reversed(sources_document['sources'])
        if item['source_id'].startswith('imi-monthly-')
    )
    stamp = imi_source['source_id'].removeprefix('imi-monthly-')
    master_source = next(
        item for item in sources_document['sources']
        if item['source_id'] == f'imi-master-{stamp}'
    )
    master_path = RAW_DIR / master_source['file']
    master_data = json.loads(master_path.read_text(encoding='utf-8'))

assert isinstance(master_data, list)
imi_master = next(item for item in master_data if item['type'] == 'IMI')
periods = imi_master['dataAvailablePeriods']
assert len(periods) == 1 and periods[0]['periodType'] == 'month'
period = periods[0]

year_base = int(period['yearBase'])
start_year_be = int(period['startYear'])
start_month = int(period['startPeriod'])
end_year_be = int(period['endYear'])
end_month = int(period['endPeriod'])
commodity_codes = [str(item['code']).zfill(3) for item in imi_master['commodities']]
months = enumerate_months(start_year_be, start_month, end_year_be, end_month)

master_summary = pd.DataFrame([{
    'ปีฐาน (พ.ศ.)': year_base,
    'เดือนเริ่มต้น': f'{start_year_be - 543:04d}-{start_month:02d}',
    'เดือนล่าสุด': f'{end_year_be - 543:04d}-{end_month:02d}',
    'จำนวนเดือน': len(months),
    'จำนวนรหัส IMI': len(commodity_codes),
}])
display(master_summary)

,ปีฐาน (พ.ศ.),เดือนเริ่มต้น,เดือนล่าสุด,จำนวนเดือน,จำนวนรหัส IMI
0,2555,2000-01,2026-08,320,81


## 3. ดาวน์โหลด IMI รายเดือน

API รับหนึ่งเดือนต่อ request และสามารถส่งรหัสสินค้าทั้งหมดพร้อมกันได้ แต่ละ request จึงประกอบด้วยปีฐาน ปี พ.ศ. เดือน ประเภท `IMI` และรหัสสินค้าทั้ง 81 รหัส มีการหน่วงเวลา 0.30 วินาทีและ retry สำหรับ HTTP 429 และ server error

In [3]:
if DOWNLOAD_NEW:
    imi_rows = []
    request_manifest = []
    for sequence, (year_be, month) in enumerate(months, start=1):
        body = {
            'yearBase': year_base,
            'year': year_be,
            'month': month,
            'type': 'IMI',
            'commodities': commodity_codes,
        }
        response = session.post(MONTH_URL, json=body, timeout=60)
        response.raise_for_status()
        payload = response.json()
        if not isinstance(payload, list) or not payload:
            raise ValueError(f'ไม่มีข้อมูล พ.ศ. {year_be}-{month:02d}')
        if any(int(row['year']) != year_be or int(row['month']) != month for row in payload):
            raise ValueError(f'เดือนในผลตอบกลับไม่ตรงกับ request: {year_be}-{month:02d}')
        imi_rows.extend(payload)
        request_manifest.append({
            'sequence': sequence, 'body': body, 'row_count': len(payload)
        })
        if sequence % 40 == 0 or sequence == len(months):
            print(f'{sequence}/{len(months)} เดือน; สะสม {len(imi_rows):,} แถว')
        if sequence < len(months):
            time.sleep(REQUEST_DELAY_SECONDS)

    manifest_path = RAW_DIR / f'imi_request_manifest_{stamp}.json'
    manifest_path.write_text(
        json.dumps(request_manifest, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    imi_df = pd.DataFrame(imi_rows)
    imi_path = RAW_DIR / f'imi_monthly_{stamp}.csv'
    imi_df.to_csv(imi_path, index=False, encoding='utf-8')
else:
    imi_path = RAW_DIR / imi_source['file']
    manifest_path = RAW_DIR / imi_source['request_bodies_file']
    imi_df = pd.read_csv(imi_path, dtype={'commodityCode': 'string'})
    imi_df['commodityCode'] = imi_df['commodityCode'].str.zfill(3)
    request_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

required_columns = {
    'commodityCode', 'commodityNameTH', 'month', 'year', 'yearBase',
    'index', 'change', 'changeYear', 'changeAVG', 'level', 'createdAt',
}
assert required_columns <= set(imi_df.columns)
assert not imi_df.duplicated(['commodityCode', 'year', 'month']).any()
returned_months = set(zip(imi_df['year'].astype(int), imi_df['month'].astype(int)))
assert returned_months == set(months)
assert len(request_manifest) == len(months)

group_coverage = (
    imi_df[imi_df['commodityCode'].isin(['000', '100', '318', '402'])]
    .groupby('commodityCode')
    .agg(rows=('index', 'size'), missing_index=('index', lambda x: x.isna().sum()))
    .reset_index()
)
assert (group_coverage['rows'] == len(months)).all()
display(group_coverage)
print(f'IMI ทั้งหมด {len(imi_df):,} แถว จาก {imi_df["commodityCode"].nunique()} รหัส')

,commodityCode,rows,missing_index
0,000,320,0
1,100,320,0
2,318,320,0
3,402,320,0


IMI ทั้งหมด 25,752 แถว จาก 81 รหัส


## 4. ดาวน์โหลดข้อมูลรายวันจาก FRED

ใช้ไฟล์ CSV ที่ไม่ต้องใช้ API key และเก็บ byte stream จากต้นทางโดยตรง ระหว่างการรันจริงพบว่า Python `requests` เกิด read timeout กับ FRED ขณะที่ `curl` ดาวน์โหลด URL เดียวกันสำเร็จ จึงใช้ `curl` พร้อม timeout และ retry สำหรับสองชุดข้อมูลนี้

In [4]:
fred_frames = {}
fred_paths = {}
target_end = pd.Timestamp(year=end_year_be - 543, month=end_month, day=1)

for series_id, url in FRED_SERIES.items():
    if DOWNLOAD_NEW:
        completed = subprocess.run(
            [
                'curl', '--fail', '--silent', '--show-error', '--location',
                '--connect-timeout', '15', '--max-time', '60',
                '--retry', '3', '--retry-all-errors', url,
            ],
            check=True, capture_output=True, timeout=75,
        )
        if not completed.stdout:
            raise ValueError(f'{series_id}: FRED ส่งไฟล์ว่างกลับมา')
        path = RAW_DIR / f'{series_id.lower()}_daily_{stamp}.csv'
        path.write_bytes(completed.stdout)
        frame = pd.read_csv(io.BytesIO(completed.stdout), na_values=['.'])
    else:
        source = next(
            item for item in sources_document['sources']
            if item['source_id'] == f'{series_id.lower()}-{stamp}'
        )
        path = RAW_DIR / source['file']
        frame = pd.read_csv(path, na_values=['.'])

    assert list(frame.columns) == ['observation_date', series_id]
    frame['observation_date'] = pd.to_datetime(frame['observation_date'], errors='raise')
    assert frame['observation_date'].is_monotonic_increasing
    assert frame['observation_date'].min() <= pd.Timestamp('2000-01-01')
    assert frame['observation_date'].max() >= target_end
    fred_frames[series_id] = frame
    fred_paths[series_id] = path

fred_summary = pd.DataFrame([
    {
        'series': series_id,
        'rows': len(frame),
        'non_null': int(frame[series_id].notna().sum()),
        'missing': int(frame[series_id].isna().sum()),
        'start': frame['observation_date'].min().date(),
        'end': frame['observation_date'].max().date(),
    }
    for series_id, frame in fred_frames.items()
])
display(fred_summary)

,series,rows,non_null,missing,start,end
0,DEXTHUS,11931,11395,536,1981-01-02,2026-09-25
1,DCOILBRENTEU,10270,9097,1173,1987-05-20,2026-09-29


## 5. บันทึก metadata และ checksum

เมื่อดาวน์โหลดชุดใหม่ จะเพิ่มรายการใน `sources.json` หลังจากข้อมูลทั้งหมดผ่าน validation แล้วเท่านั้น แต่ละรายการระบุ URL, method, request body หรือ manifest, เวลาดาวน์โหลด, จำนวนแถว, ช่วงข้อมูล, ใบอนุญาต และ SHA-256

In [5]:
if DOWNLOAD_NEW:
    downloaded_at_iso = downloaded_at.isoformat().replace('+00:00', 'Z')
    new_records = [
        {
            'source_id': f'imi-master-{stamp}',
            'name': 'IMI MasterData',
            'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'url': MASTER_URL, 'method': 'GET',
            'downloaded_at_utc': downloaded_at_iso,
            'row_count': len(master_data), 'data_range': None,
            'file': master_path.name, 'sha256': file_sha256(master_path),
            'license': 'Open Data Common',
            'citation': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. ข้อมูลดัชนีราคาสินค้านำเข้าของประเทศ. ศูนย์กลางข้อมูลเปิดภาครัฐ.',
        },
        {
            'source_id': f'imi-monthly-{stamp}',
            'name': 'ดัชนีราคาสินค้านำเข้ารายเดือน (IMI)',
            'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'url': MONTH_URL, 'method': 'POST',
            'request_body_template': {
                'yearBase': year_base, 'year': '<พ.ศ.>', 'month': '<1-12>',
                'type': 'IMI', 'commodities': commodity_codes,
            },
            'request_bodies_file': manifest_path.name,
            'request_bodies_sha256': file_sha256(manifest_path),
            'request_count': len(request_manifest), 'failed_request_count': 0,
            'downloaded_at_utc': downloaded_at_iso, 'row_count': len(imi_df),
            'data_range': {
                'start': f'{start_year_be - 543:04d}-{start_month:02d}',
                'end': f'{end_year_be - 543:04d}-{end_month:02d}',
                'calendar_in_file': 'Buddhist Era',
            },
            'file': imi_path.name, 'sha256': file_sha256(imi_path),
            'license': 'Open Data Common',
            'citation': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. ข้อมูลดัชนีราคาสินค้านำเข้าของประเทศ. ศูนย์กลางข้อมูลเปิดภาครัฐ.',
        },
    ]
    fred_metadata = {
        'DEXTHUS': ('Thai Baht to U.S. Dollar Spot Exchange Rate', 'Board of Governors of the Federal Reserve System (US)'),
        'DCOILBRENTEU': ('Crude Oil Prices: Brent - Europe', 'U.S. Energy Information Administration'),
    }
    for series_id, frame in fred_frames.items():
        name, owner = fred_metadata[series_id]
        new_records.append({
            'source_id': f'{series_id.lower()}-{stamp}', 'name': name,
            'owner': owner, 'publisher': 'FRED, Federal Reserve Bank of St. Louis',
            'url': FRED_SERIES[series_id], 'method': 'GET',
            'downloaded_at_utc': downloaded_at_iso, 'row_count': len(frame),
            'non_null_row_count': int(frame[series_id].notna().sum()),
            'data_range': {
                'start': str(frame['observation_date'].min().date()),
                'end': str(frame['observation_date'].max().date()),
            },
            'file': fred_paths[series_id].name,
            'sha256': file_sha256(fred_paths[series_id]),
            'license': 'Public Domain: Citation Requested',
            'citation': f'{owner}, {name} [{series_id}], retrieved from FRED, Federal Reserve Bank of St. Louis.',
        })
    sources_document['sources'].extend(new_records)
    sources_document['last_updated_at_utc'] = downloaded_at_iso
    SOURCES_PATH.write_text(
        json.dumps(sources_document, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
else:
    new_records = [
        item for item in sources_document['sources']
        if item['source_id'].endswith(stamp)
    ]

metadata_summary = pd.DataFrame([
    {
        'source_id': item['source_id'],
        'file': item['file'],
        'rows': item['row_count'],
        'sha256': item['sha256'][:12] + '...',
    }
    for item in new_records
])
display(metadata_summary)

,source_id,file,rows,sha256
0,imi-master-20261005T054207Z,imi_master_20261005T054207Z.json,2,618ad4c6de8d...
1,imi-monthly-20261005T054207Z,imi_monthly_20261005T054207Z.csv,25752,55a338b42c21...
2,dexthus-20261005T054207Z,dexthus_daily_20261005T054207Z.csv,11931,82b99fc8c2e1...
3,dcoilbrenteu-20261005T054207Z,dcoilbrenteu_daily_20261005T054207Z.csv,10270,68b1b12a3e21...


## 6. Checkpoint ก่อนเข้าสู่ Data Understanding

เงื่อนไขผ่านคือ IMI ครบทุกเดือนตาม MasterData ไม่มี request ล้มเหลว กลุ่มหลักทั้งสี่รหัสมีข้อมูลครบ ข้อมูล FRED ครอบคลุมตั้งแต่ปี 2000 ถึงเดือนล่าสุดของ IMI และ checksum ตรงกับ `sources.json`

In [6]:
record_by_file = {item['file']: item for item in new_records}
files_to_verify = [master_path, imi_path, *fred_paths.values()]
checksum_results = []
for path in files_to_verify:
    expected = record_by_file[path.name]['sha256']
    actual = file_sha256(path)
    checksum_results.append({
        'file': path.name, 'checksum_match': actual == expected
    })
checksum_df = pd.DataFrame(checksum_results)
display(checksum_df)

assert checksum_df['checksum_match'].all()
assert len(months) == len(request_manifest)
assert (group_coverage['rows'] == len(months)).all()
assert all(frame['observation_date'].max() >= target_end for frame in fred_frames.values())

checkpoint = pd.DataFrame([{
    'IMI rows': len(imi_df),
    'IMI codes': imi_df['commodityCode'].nunique(),
    'IMI months': len(months),
    'Failed requests': 0,
    'DEXTHUS rows': len(fred_frames['DEXTHUS']),
    'Brent rows': len(fred_frames['DCOILBRENTEU']),
    'Checksum files passed': int(checksum_df['checksum_match'].sum()),
}])
display(checkpoint)
print('CHECKPOINT PASSED: ข้อมูลทั้งสามแหล่งครบตามเงื่อนไขของโครงการ')

if DOWNLOAD_NEW:
    fx = fred_frames['DEXTHUS']
    brent = fred_frames['DCOILBRENTEU']
    log_entry = f'''
## [{downloaded_at.date().isoformat()}] ขั้นตอน: 01 Data Acquisition
**สิ่งที่ทำ:** ดาวน์โหลด IMI ทุกรหัสตาม MasterData จำนวน {len(months):,} เดือน ดาวน์โหลด DEXTHUS และ DCOILBRENTEU จาก FRED ตรวจ schema และบันทึก metadata พร้อม SHA-256

**ผลที่ได้:** IMI จำนวน {len(imi_df):,} แถว ครอบคลุม {start_year_be - 543:04d}-{start_month:02d} ถึง {end_year_be - 543:04d}-{end_month:02d} จาก {len(commodity_codes):,} รหัสและ {len(request_manifest):,} requests โดยไม่มี request ล้มเหลว; DEXTHUS จำนวน {len(fx):,} แถว ช่วง {fx['observation_date'].min().date()} ถึง {fx['observation_date'].max().date()}; DCOILBRENTEU จำนวน {len(brent):,} แถว ช่วง {brent['observation_date'].min().date()} ถึง {brent['observation_date'].max().date()}

**สิ่งที่พบ / ปัญหา:** เก็บค่าว่างของข้อมูลรายวันตามต้นฉบับและไม่แทนด้วยศูนย์

**การตัดสินใจและเหตุผล:** เก็บ raw file ด้วย timestamp UTC และ request manifest แยกต่างหากเพื่อป้องกันการเขียนทับและตรวจสอบย้อนกลับได้

**ขั้นต่อไป:** ดำเนินการ 02 Data Understanding
'''
    with (ROOT / 'logs' / 'process_log.md').open('a', encoding='utf-8') as handle:
        handle.write(log_entry)
    print('บันทึกผลรอบใหม่ลง logs/process_log.md แล้ว')

,file,checksum_match
0,imi_master_20261005T054207Z.json,True
1,imi_monthly_20261005T054207Z.csv,True
2,dexthus_daily_20261005T054207Z.csv,True
3,dcoilbrenteu_daily_20261005T054207Z.csv,True


,IMI rows,IMI codes,IMI months,Failed requests,DEXTHUS rows,Brent rows,Checksum files passed
0,25752,81,320,0,11931,10270,4


CHECKPOINT PASSED: ข้อมูลทั้งสามแหล่งครบตามเงื่อนไขของโครงการ


## 7. Process log

ผลการดาวน์โหลดจริง จำนวนแถว ปัญหา FRED timeout และเหตุผลที่เลือกวิธีแก้ไข ถูกบันทึกไว้ใน `logs/process_log.md` แล้ว การเปิด notebook ในโหมดอ่าน snapshot จะไม่เขียน log ซ้ำ ส่วนโหมดดาวน์โหลดใหม่จะเขียนผลต่อท้ายโดยอัตโนมัติหลังผ่าน checkpoint

## 8. ปฏิทินเผยแพร่ IMI จากข่าว TPSO

หลังตรวจพบว่าวันที่ 25 ของเดือนถัดไปไม่ตรงกับวันเผยแพร่จริง จึงเก็บวันเผยแพร่จาก archive ข่าวประชาสัมพันธ์ TPSO ทุกหน้าที่มีอยู่ และเลือกข่าวที่ระบุเดือนอ้างอิงของดัชนีราคาส่งออก-นำเข้าอย่างชัดเจน

ตั้งค่า `DOWNLOAD_RELEASE_CALENDAR = True` เมื่อต้องการดึง archive ใหม่ ค่าเริ่มต้นอ่าน snapshot ที่บันทึกไว้เพื่อไม่ยิงเว็บไซต์ซ้ำ

In [7]:
import re
from bs4 import BeautifulSoup

DOWNLOAD_RELEASE_CALENDAR = False
CATEGORY_URL = 'https://www.tpso.go.th/category/%E0%B8%82%E0%B9%88%E0%B8%B2%E0%B8%A7%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B8%8A%E0%B8%B2%E0%B8%AA%E0%B8%B1%E0%B8%A1%E0%B8%9E%E0%B8%B1%E0%B8%99%E0%B8%98%E0%B9%8C?page={page}'
THAI_MONTHS = {
    'มกราคม': 1, 'กุมภาพันธ์': 2, 'มีนาคม': 3, 'เมษายน': 4,
    'พฤษภาคม': 5, 'มิถุนายน': 6, 'กรกฎาคม': 7, 'สิงหาคม': 8,
    'กันยายน': 9, 'ตุลาคม': 10, 'พฤศจิกายน': 11, 'ธันวาคม': 12,
}

if DOWNLOAD_RELEASE_CALENDAR:
    release_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    first_response = session.get(CATEGORY_URL.format(page=1), timeout=60)
    first_response.raise_for_status()
    first_data = json.loads(BeautifulSoup(first_response.text, 'html.parser').find('script', id='__NEXT_DATA__').string)
    result = first_data['props']['pageProps']['result']
    page_count = int(result['pagecount'])
    news_rows = list(result['data'])
    request_manifest_release = [{'page': 1, 'url': CATEGORY_URL.format(page=1), 'row_count': len(result['data'])}]
    for page in range(2, page_count + 1):
        response = session.get(CATEGORY_URL.format(page=page), timeout=60)
        response.raise_for_status()
        page_result = json.loads(BeautifulSoup(response.text, 'html.parser').find('script', id='__NEXT_DATA__').string)['props']['pageProps']['result']
        news_rows.extend(page_result['data'])
        request_manifest_release.append({'page': page, 'url': CATEGORY_URL.format(page=page), 'row_count': len(page_result['data'])})

    release_rows = []
    pattern = re.compile(r'เดือน(มกราคม|กุมภาพันธ์|มีนาคม|เมษายน|พฤษภาคม|มิถุนายน|กรกฎาคม|สิงหาคม|กันยายน|ตุลาคม|พฤศจิกายน|ธันวาคม)\s+(25\d{2})')
    for item in news_rows:
        title = item.get('NewsTitleTh') or ''
        is_imi_news = ('ดัชนีราคาส่งออก-นำเข้า' in title or 'ดัชนีราคาส่งออก และดัชนีราคานำเข้า' in title)
        match = pattern.search(title) if is_imi_news else None
        if not match:
            continue
        reference_month = pd.Timestamp(int(match.group(2)) - 543, THAI_MONTHS[match.group(1)], 1)
        release_datetime = pd.Timestamp(item['PublishStartTime'])
        release_rows.append({
            'reference_month': reference_month,
            'release_datetime': release_datetime,
            'lag_days_after_month_end': (release_datetime.normalize() - (reference_month + pd.offsets.MonthEnd(0))).days,
            'news_code': item['NewsCode'],
            'title_th': title.strip(),
            'url': f"https://www.tpso.go.th/news/{item['NewsCode']}",
        })
    release_calendar = pd.DataFrame(release_rows).sort_values('reference_month').drop_duplicates('reference_month', keep='last')
    release_path = RAW_DIR / f'tpso_imi_release_calendar_{release_stamp}.csv'
    release_manifest_path = RAW_DIR / f'tpso_news_request_manifest_{release_stamp}.json'
    release_calendar.to_csv(release_path, index=False)
    release_manifest_path.write_text(json.dumps(request_manifest_release, ensure_ascii=False, indent=2), encoding='utf-8')

    sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))
    source_record = {
        'source_id': f'tpso-release-calendar-{release_stamp}',
        'name': 'วันเผยแพร่ข่าวดัชนีราคาส่งออก-นำเข้าของไทย',
        'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
        'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
        'url': CATEGORY_URL.replace('{page}', '<page>'),
        'method': 'GET',
        'request_count': page_count,
        'request_manifest_file': release_manifest_path.name,
        'request_manifest_sha256': file_sha256(release_manifest_path),
        'downloaded_at_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
        'row_count': len(release_calendar),
        'data_range': {
            'start': str(release_calendar['reference_month'].min().date()),
            'end': str(release_calendar['reference_month'].max().date()),
        },
        'file': release_path.name,
        'sha256': file_sha256(release_path),
        'license': '[ยังไม่ยืนยัน]',
        'citation': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. ข่าวประชาสัมพันธ์ดัชนีราคาส่งออก-นำเข้าของไทย.',
    }
    sources_document['sources'].append(source_record)
    sources_document['last_updated_at_utc'] = source_record['downloaded_at_utc']
    SOURCES_PATH.write_text(json.dumps(sources_document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
else:
    sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))
    source_record = next(item for item in reversed(sources_document['sources']) if item['source_id'].startswith('tpso-release-calendar-'))
    release_path = RAW_DIR / source_record['file']
    release_calendar = pd.read_csv(release_path, parse_dates=['reference_month', 'release_datetime'])

assert file_sha256(release_path) == source_record['sha256']
assert not release_calendar['reference_month'].duplicated().any()
release_calendar['release_day_in_t_plus_2'] = (
    release_calendar['release_datetime'].dt.normalize()
    - (release_calendar['reference_month'] + pd.offsets.MonthBegin(2))
).dt.days + 1
display(release_calendar[['reference_month', 'release_datetime', 'lag_days_after_month_end', 'news_code']])
display(release_calendar[['lag_days_after_month_end', 'release_day_in_t_plus_2']].describe())
print(f"ตรวจพบ {len(release_calendar)} เดือน; วันเผยแพร่ช้าที่สุดคือวันที่ {int(release_calendar['release_day_in_t_plus_2'].max())} ของเดือน t+2")

,reference_month,release_datetime,lag_days_after_month_end,news_code
0,2024-11-01,2025-01-04 15:30:00,35,2501-0000000002
1,2024-12-01,2025-01-28 15:30:00,28,2501-0000000017
2,2025-01-01,2025-02-27 15:30:00,27,2502-0000000039
3,2025-02-01,2025-03-27 15:30:00,27,2503-0000000017
4,2025-03-01,2025-04-30 15:30:00,30,2504-0000000014
5,2025-04-01,2025-05-29 15:30:00,29,2505-0000000013
6,2025-05-01,2025-06-26 15:30:00,26,2506-0000000019
7,2025-06-01,2025-07-29 15:30:00,29,2508-0000000002
8,2025-07-01,2025-08-28 15:30:00,28,2508-0000000021
9,2025-08-01,2025-09-29 15:30:00,29,2509-0000000013


,lag_days_after_month_end,release_day_in_t_plus_2
count,21.000000,21.000000
mean,31.190476,0.761905
std,4.578417,4.592437
min,26.000000,-4.000000
25%,28.000000,-3.000000
50%,29.000000,-1.000000
75%,34.000000,3.000000
max,41.000000,10.000000


ตรวจพบ 21 เดือน; วันเผยแพร่ช้าที่สุดคือวันที่ 10 ของเดือน t+2


## 9. ดัชนีราคาในประเทศสำหรับวิเคราะห์ pass-through

IMI เป็นราคา C.I.F. ในรูปดอลลาร์สหรัฐ ขณะที่ผู้ใช้ซึ่งเป็น SME จ่ายราคาในประเทศเป็นบาท จึงดึงดัชนีราคาในประเทศของ สนค. จาก API เดียวกับ IMI เพื่อตรวจว่าราคาในประเทศตามหลัง IMI กี่เดือน (Scope หัวข้อ 5.4 และ RQ4)

- **CPI นมและผลิตภัณฑ์นม** (รหัส `11320`, ระดับประเทศ `TG`, ปีฐาน 2566): ส่ง request ตั้งแต่ ม.ค. 2543 แต่ API มีข้อมูลรายการนี้ตั้งแต่ ธ.ค. 2544 (23 request แรกได้ผลว่าง)
- **PPI ผลิตภัณฑ์นม** (`3105`) และ **PPI เคมีภัณฑ์ขั้นมูลฐาน ปุ๋ยเคมีและสารประกอบไนโตรเจน พลาสติกและยางสังเคราะห์ขั้นต้น** (`3201`) (ประเภท `CPA`, ปีฐาน 2564): มีข้อมูลตั้งแต่ ม.ค. 2558 รหัส 3201 รวมปุ๋ยกับพลาสติก จึงใช้แทนราคาปุ๋ยได้เพียงคร่าว ๆ

ถ้ายังไม่มี snapshot ใน `sources.json` cell นี้จะดาวน์โหลดเอง ถ้ามีแล้วจะอ่าน snapshot เดิม

In [8]:
DOWNLOAD_DOMESTIC = False
DOMESTIC_SERIES = [
    {'key': 'cpi', 'name': 'ดัชนีราคาผู้บริโภคทั่วไประดับประเทศ (CPI)',
     'master_url': 'https://index-api.tpso.go.th/OpenApi/Cpig/Month/MasterData',
     'url': 'https://index-api.tpso.go.th/OpenApi/Cpig/Month',
     'type': 'TG', 'commodities': ['11320'], 'request_start': (2543, 1)},
    {'key': 'ppi', 'name': 'ดัชนีราคาผู้ผลิต (PPI) จำแนกตาม CPA',
     'master_url': 'https://index-api.tpso.go.th/OpenApi/Ppi/Month/MasterData',
     'url': 'https://index-api.tpso.go.th/OpenApi/Ppi/Month',
     'type': 'CPA', 'commodities': ['3105', '3201'], 'request_start': None},
]

sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))
domestic_frames, domestic_records = {}, []
for spec in DOMESTIC_SERIES:
    prefix = f"tpso-{spec['key']}-monthly-"
    existing = [item for item in sources_document['sources'] if item['source_id'].startswith(prefix)]
    if not DOWNLOAD_DOMESTIC and not existing:
        raise FileNotFoundError(f"ไม่พบ snapshot {prefix}* ใน sources.json; ตั้ง DOWNLOAD_DOMESTIC = True เพื่อดาวน์โหลดใหม่โดยตั้งใจ")
    if DOWNLOAD_DOMESTIC:
        dom_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        master_response = session.get(spec['master_url'], timeout=60)
        master_response.raise_for_status()
        master_json = master_response.json()
        master_block = next(
            block for block in (master_json if isinstance(master_json, list) else [master_json])
            if spec['key'] == 'cpi' or block.get('type') == spec['type']
        )
        latest_period = max(master_block['dataAvailablePeriods'], key=lambda item: int(item['yearBase']))
        dom_year_base = int(latest_period['yearBase'])
        start = spec['request_start'] or (int(latest_period['startYear']), int(latest_period['startPeriod']))
        dom_months = enumerate_months(*start, int(latest_period['endYear']), int(latest_period['endPeriod']))
        rows, manifest = [], []
        for sequence, (year_be, month) in enumerate(dom_months, start=1):
            body = {'yearBase': dom_year_base, 'year': year_be, 'month': month,
                    'type': spec['type'], 'commodities': spec['commodities']}
            response = session.post(spec['url'], json=body, timeout=60)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, list):
                raise ValueError(f"{spec['key']}: ผลตอบกลับไม่ใช่ list ที่ {year_be}-{month:02d}: {payload}")
            rows.extend(payload)
            manifest.append({'sequence': sequence, 'body': body, 'row_count': len(payload)})
            if sequence < len(dom_months):
                time.sleep(REQUEST_DELAY_SECONDS)
        frame = pd.DataFrame(rows)
        data_path = RAW_DIR / f"tpso_{spec['key']}_monthly_{dom_stamp}.csv"
        manifest_path_dom = RAW_DIR / f"tpso_{spec['key']}_request_manifest_{dom_stamp}.json"
        frame.to_csv(data_path, index=False, encoding='utf-8')
        manifest_path_dom.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        record = {
            'source_id': f'{prefix}{dom_stamp}', 'name': spec['name'],
            'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'url': spec['url'], 'method': 'POST',
            'request_body_template': {'yearBase': dom_year_base, 'year': '<พ.ศ.>', 'month': '<1-12>',
                                      'type': spec['type'], 'commodities': spec['commodities']},
            'request_bodies_file': manifest_path_dom.name,
            'request_bodies_sha256': file_sha256(manifest_path_dom),
            'request_count': len(manifest), 'failed_request_count': 0,
            'empty_response_count': sum(item['row_count'] == 0 for item in manifest),
            'downloaded_at_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
            'row_count': len(frame),
            'data_range': {'start': f"{int(frame.year.min()) - 543}-{int(frame.loc[frame.year == frame.year.min(), 'month'].min()):02d}",
                           'end': f"{int(frame.year.max()) - 543}-{int(frame.loc[frame.year == frame.year.max(), 'month'].max()):02d}",
                           'calendar_in_file': 'Buddhist Era'},
            'file': data_path.name, 'sha256': file_sha256(data_path),
            'license': 'Open Data Common [ยังไม่ยืนยันหน้าชุดข้อมูลบน data.go.th ของรายการนี้]',
            'citation': f"สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. {spec['name']}. index.tpso.go.th.",
        }
        sources_document['sources'].append(record)
        sources_document['last_updated_at_utc'] = record['downloaded_at_utc']
        SOURCES_PATH.write_text(json.dumps(sources_document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    else:
        record = existing[-1]
        data_path = RAW_DIR / record['file']
        frame = pd.read_csv(data_path, dtype={'commodityCode': 'string'})
    assert file_sha256(RAW_DIR / record['file']) == record['sha256']
    frame['commodityCode'] = frame['commodityCode'].astype('string')
    frame['date'] = pd.to_datetime(dict(year=frame['year'].astype(int) - 543, month=frame['month'].astype(int), day=1))
    assert not frame.duplicated(['commodityCode', 'date']).any()
    for code, group in frame.groupby('commodityCode'):
        expected = pd.date_range(group['date'].min(), group['date'].max(), freq='MS')
        assert len(group) == len(expected), f"{spec['key']} {code}: เดือนไม่ต่อเนื่อง"
        assert group['index'].notna().all() and (group['index'] > 0).all()
    domestic_frames[spec['key']] = frame
    domestic_records.append(record)

domestic_summary = pd.concat([
    frame.groupby(['commodityCode', 'commodityNameTH']).agg(
        rows=('index', 'size'), start=('date', 'min'), end=('date', 'max'),
        year_base=('yearBase', 'first')).reset_index().assign(source=key)
    for key, frame in domestic_frames.items()
], ignore_index=True)
display(domestic_summary)
display(pd.DataFrame([{'source_id': r['source_id'], 'file': r['file'], 'rows': r['row_count'],
                       'requests': r['request_count'], 'empty_responses': r['empty_response_count'],
                       'sha256': r['sha256'][:12] + '...'} for r in domestic_records]))

,commodityCode,commodityNameTH,rows,start,end,year_base,source
0,11320,นมและผลิตภัณฑ์นม,297,2001-12-01,2026-08-01,2566,cpi
1,3105,ผลิตภัณฑ์นม,140,2015-01-01,2026-08-01,2564,ppi
2,3201,เคมีภัณฑ์ขั้นมูลฐาน ปุ๋ยเคมีและสารประกอบไนโตรเ...,140,2015-01-01,2026-08-01,2564,ppi


,source_id,file,rows,requests,empty_responses,sha256
0,tpso-cpi-monthly-20261005T083615Z,tpso_cpi_monthly_20261005T083615Z.csv,297,320,23,92a43bdf5743...
1,tpso-ppi-monthly-20261005T084014Z,tpso_ppi_monthly_20261005T084014Z.csv,280,140,0,bac4de3ebde2...


## 10. ราคาวัตถุดิบโลกสำหรับรอบที่ 2 (Scope หัวข้อ 24)

ข้อมูลส่วนนี้ใช้ในรอบที่ 2 ซึ่งเป็นผลเชิงสำรวจหลังเปิดผล test แล้ว
- **World Bank Pink Sheet:** ไฟล์ Excel ราคารายเดือน ใช้ series `Urea` และ `DAP` (USD/ตัน) ใบอนุญาต CC BY 4.0
- **FAO Food Price Index:** ไฟล์ CSV ดัชนีรายเดือน ใช้ series `Dairy` (2014–2016 = 100)

ดาวน์โหลดด้วย `curl` พร้อม timeout และ retry แบบเดียวกับ FRED ถ้ามี snapshot ใน `sources.json` แล้วจะอ่าน snapshot เดิม

In [9]:
DOWNLOAD_COMMODITY = False
COMMODITY_SOURCES = [
    {'key': 'wb-pinksheet', 'name': 'World Bank Commodity Price Data (The Pink Sheet), monthly prices',
     'owner': 'World Bank', 'suffix': 'xlsx', 'license': 'CC BY 4.0',
     'url': 'https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx',
     'citation': 'World Bank. Commodity Price Data (The Pink Sheet), monthly prices in nominal US dollars.'},
    {'key': 'fao-ffpi', 'name': 'FAO Food Price Index, monthly nominal indices',
     'owner': 'Food and Agriculture Organization of the United Nations (FAO)', 'suffix': 'csv',
     'license': '[ยังไม่ยืนยัน] ตาม FAO Terms https://www.fao.org/contact-us/terms/en/',
     'url': 'https://www.fao.org/media/docs/worldfoodsituationlibraries/wfs-library/food_price_indices_data.csv',
     'citation': 'FAO. FAO Food Price Index, Dairy Price Index (2014-2016 = 100).'},
]

def load_pinksheet(path):
    sheet = pd.read_excel(path, sheet_name='Monthly Prices', header=None)
    names = [str(x).strip() if pd.notna(x) else '' for x in sheet.iloc[4]]
    cols = {name: names.index(name) for name in ('Urea', 'DAP')}
    frame = sheet.iloc[6:, [0, *cols.values()]].copy()
    frame.columns = ['period', *cols.keys()]
    frame = frame[frame.period.astype(str).str.match(r'^\d{4}M\d{2}$')]
    frame['month_ce'] = pd.to_datetime(frame.period.str.replace('M', '-') + '-01')
    for name in cols: frame[name] = pd.to_numeric(frame[name], errors='coerce')
    return frame[['month_ce', *cols.keys()]].reset_index(drop=True)

def load_fao(path):
    raw_lines = Path(path).read_text(encoding='utf-8-sig').splitlines()
    header = next(i for i, line in enumerate(raw_lines) if line.startswith('Date,'))
    frame = pd.read_csv(path, skiprows=header, encoding='utf-8-sig').dropna(subset=['Date'])
    frame = frame[frame.Date.astype(str).str.match(r'^\d{4}-\d{2}$')]
    frame['month_ce'] = pd.to_datetime(frame.Date + '-01')
    frame['Dairy'] = pd.to_numeric(frame.Dairy, errors='coerce')
    return frame[['month_ce', 'Dairy']].reset_index(drop=True)

sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))
commodity_frames, commodity_records = {}, []
for spec in COMMODITY_SOURCES:
    prefix = f"{spec['key']}-monthly-"
    existing = [item for item in sources_document['sources'] if item['source_id'].startswith(prefix)]
    if not DOWNLOAD_COMMODITY and not existing:
        raise FileNotFoundError(f"ไม่พบ snapshot {prefix}* ใน sources.json; ตั้ง DOWNLOAD_COMMODITY = True เพื่อดาวน์โหลดใหม่โดยตั้งใจ")
    if DOWNLOAD_COMMODITY:
        com_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        completed = subprocess.run(
            ['curl', '--fail', '--silent', '--show-error', '--location', '--connect-timeout', '15',
             '--max-time', '120', '--retry', '3', '--retry-all-errors', spec['url']],
            check=True, capture_output=True, timeout=150)
        if not completed.stdout:
            raise ValueError(f"{spec['key']}: ได้ไฟล์ว่าง")
        path = RAW_DIR / f"{spec['key'].replace('-', '_')}_monthly_{com_stamp}.{spec['suffix']}"
        path.write_bytes(completed.stdout)
        frame = load_pinksheet(path) if spec['key'] == 'wb-pinksheet' else load_fao(path)
        record = {
            'source_id': f'{prefix}{com_stamp}', 'name': spec['name'], 'owner': spec['owner'],
            'publisher': spec['owner'], 'url': spec['url'], 'method': 'GET',
            'downloaded_at_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
            'row_count': len(frame), 'series_used': [c for c in frame.columns if c != 'month_ce'],
            'data_range': {'start': str(frame.month_ce.min().date()), 'end': str(frame.month_ce.max().date())},
            'file': path.name, 'sha256': file_sha256(path), 'license': spec['license'], 'citation': spec['citation'],
            'note': 'ใช้ในรอบที่ 2 (Scope หัวข้อ 24) ซึ่งเป็นผลเชิงสำรวจหลังเปิดผล test',
        }
        sources_document['sources'].append(record)
        sources_document['last_updated_at_utc'] = record['downloaded_at_utc']
        SOURCES_PATH.write_text(json.dumps(sources_document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    else:
        record = existing[-1]
        path = RAW_DIR / record['file']
        frame = load_pinksheet(path) if spec['key'] == 'wb-pinksheet' else load_fao(path)
    assert file_sha256(RAW_DIR / record['file']) == record['sha256']
    since_2000 = frame[frame.month_ce >= '2000-01-01']
    assert since_2000.month_ce.is_monotonic_increasing and not since_2000.month_ce.duplicated().any()
    assert len(since_2000) == len(pd.date_range('2000-01-01', since_2000.month_ce.max(), freq='MS'))
    assert since_2000.drop(columns='month_ce').notna().all().all() and (since_2000.drop(columns='month_ce') > 0).all().all()
    assert since_2000.month_ce.max() >= target_end
    commodity_frames[spec['key']] = frame
    commodity_records.append(record)

display(pd.DataFrame([{
    'source_id': r['source_id'], 'series': ', '.join(r['series_used']), 'rows': r['row_count'],
    'start': r['data_range']['start'], 'end': r['data_range']['end'], 'license': r['license'][:30],
    'sha256': r['sha256'][:12] + '...'} for r in commodity_records]))

,source_id,series,rows,start,end,license,sha256
0,wb-pinksheet-monthly-20261005T114130Z,"Urea, DAP",801,1960-01-01,2026-09-01,CC BY 4.0,ea1c35082787...
1,fao-ffpi-monthly-20261005T114131Z,Dairy,441,1990-01-01,2026-09-01,[ยังไม่ยืนยัน] ตาม FAO Terms h,1e7b4cab727d...
